\title{}
\author{}
\date{}
\makeatletter
\renewcommand{\maketitle}{}
\makeatother

\thispagestyle{empty}

\begin{center}
\vspace*{4cm}

{\LARGE Asset Allocation \& Investment Strategies \\[0.5cm]}

Academic year: 2025--2026\\[1.5cm]

Group 8\\[0.3cm]
Sacha Mimoun\\
Isabelle Chuah\\
Victor Lotigie\\
Evelyn Wang\\
Bolun Tian\\[1.5cm]

\textit{Imperial College Business School}

\end{center}

\newpage

\setcounter{secnumdepth}{0}

\thispagestyle{empty}
\clearpage

\tableofcontents

\newpage

<CENTER>
<p><font size="5"> ASSET ALLOCATION </span></p>
<p><font size="5"> ASSIGNMENT 2 : FAMA-FRENCH FACTORS (PAPER 1993) </font></p>
</p>
</CENTER>

In [54]:
import pandas as pd
import datetime
import os
import sys

In [86]:
# Import raw data

path = "raw_data/"
filename_factors = 'F-F_Research_Data_Factors.CSV'
filename_portfolios = '25_Portfolios_5x5.CSV'
filepath = os.path.join(path, filename_factors)
filepath_portfolios = os.path.join(path, filename_portfolios)

In [91]:
# Read F-F factors file - extract only monthly data
df_temp = pd.read_csv(filepath, skiprows=3, header=None)
separator_idx = df_temp[df_temp[0].astype(str).str.contains('Annual', case=False, na=False)].index[0]



In [93]:
# Read only monthly data (stop before annual section)
df_ff_monthly = pd.read_csv(filepath, skiprows=3, nrows=separator_idx-1)
# Clean monthly factors data
df_ff_monthly['date_str'] = df_ff_monthly.iloc[:, 0].astype(str).str.strip()
df_ff_monthly = df_ff_monthly[df_ff_monthly['date_str'].str.match(r'^\d{6}$')]
df_ff_monthly.index = pd.to_datetime(df_ff_monthly['date_str'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_ff_monthly = df_ff_monthly.iloc[:, 1:-1]

print("Fama-French Monthly Factors:")
print(df_ff_monthly)
print(f"Shape: {df_ff_monthly.shape}")

Fama-French Monthly Factors:
            Mkt-RF   SMB   HML    RF
date_str                            
1926-07-01    2.96 -2.56 -2.43  0.22
1926-08-01    2.64 -1.17  3.82  0.25
1926-09-01    0.36 -1.40  0.13  0.23
1926-10-01   -3.24 -0.09  0.70  0.32
1926-11-01    2.53 -0.10 -0.51  0.31
...            ...   ...   ...   ...
2023-08-01   -2.39 -3.16 -1.06  0.45
2023-09-01   -5.24 -2.51  1.52  0.43
2023-10-01   -3.19 -3.87  0.19  0.47
2023-11-01    8.84 -0.02  1.64  0.44
2023-12-01    4.85  6.35  4.94  0.43

[1170 rows x 4 columns]
Shape: (1170, 4)


The dataframes shape are consistent because 97 years is equivalent to 97*12 = 1164 but one must add the 6 last months of 1926 : 1170.

Let's do the same for the 25 portfolios

In [94]:
# Read 25 Portfolios file - extract only monthly datasets
with open(filepath_portfolios, 'r') as f:
    lines = f.readlines()

# Find indices for monthly datasets
vw_idx = next(i for i, line in enumerate(lines) if 'Average Value Weighted Returns -- Monthly' in line)
ew_idx = next(i for i, line in enumerate(lines) if 'Average Equal Weighted Returns -- Monthly' in line)

print(f"VW Monthly: line {vw_idx}, EW Monthly: line {ew_idx}")

VW Monthly: line 14, EW Monthly: line 1188


In [98]:
# Extract monthly datasets
df_port_vw = pd.read_csv(filepath_portfolios, skiprows=vw_idx+1, nrows=ew_idx-vw_idx-3)
df_port_vw['Date'] = df_port_vw.iloc[:, 0].astype(str).str.strip()
df_port_vw = df_port_vw[df_port_vw['Date'].str.match(r'^\d{6}$', na=False)]
df_port_vw.index = pd.to_datetime(df_port_vw['Date'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_port_vw = df_port_vw.iloc[:, 1:-1]

df_port_ew = pd.read_csv(filepath_portfolios, skiprows=ew_idx+1, nrows=ew_idx-vw_idx-3)
df_port_ew['Date'] = df_port_ew.iloc[:, 0].astype(str).str.strip()
df_port_ew = df_port_ew[df_port_ew['Date'].str.match(r'^\d{6}$', na=False)]
df_port_ew.index = pd.to_datetime(df_port_ew['Date'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_port_ew = df_port_ew.iloc[:, 1:-1]

print(f"\nVW Monthly: {df_port_vw.shape}")
print(df_port_ew)

print(f"EW Monthly: {df_port_ew.shape}")

print(df_port_vw)


VW Monthly: (1170, 25)
            SMALL LoBM  ME1 BM2  ME1 BM3  ME1 BM4  SMALL HiBM  ME2 BM1  \
Date                                                                     
1926-07-01      6.6071  -4.0865  -0.0568   1.9409     -1.3991   1.6885   
1926-08-01     -0.2185  -5.0680   0.6449   3.0744      6.0112   1.8012   
1926-09-01     -8.4180  -3.7750  -3.8560  -5.8874      4.9205  -3.5258   
1926-10-01     -8.3320  -4.2445  -6.9919   2.3415     -3.6773  -4.5496   
1926-11-01      0.7153   5.4507   1.5470  -3.6597      2.5410  -0.4564   
...                ...      ...      ...      ...         ...      ...   
2023-08-01     -8.7126  -9.9641 -11.9581  -6.0369     -7.6521  -6.6579   
2023-09-01     -9.8163 -10.7012  -8.3837  -5.5651     -7.2454  -9.2120   
2023-10-01     -8.5168 -10.8053  -8.2348  -8.7652     -8.2942 -11.4652   
2023-11-01      6.7387   4.8212  17.2671  10.1158      5.1756  11.2112   
2023-12-01      9.9094  17.9029  11.8879  16.5611     12.9704  13.6147   

            M